# 📊 Step 2: Exploratory Data Analysis (EDA)

## Overview
Comprehensive exploratory analysis of the supply chain dataset.

**Objectives:**
- Analyze distributions and patterns
- Identify correlations and relationships
- Explore temporal trends
- Assess data quality issues
- Generate insights for feature engineering

---


In [ ]:
# ============================================================================
# SETUP
# ============================================================================
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data.data_manager import load_raw

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

df = load_raw()
print(f"✅ Loaded {df.shape[0]:,} records")


## 2.1 Missing Values Analysis


In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

print("MISSING VALUES:")
print(missing_df)

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['Percent'].plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n💡 INTERPRETATION: Only 2-3 columns have significant missing values.")


## 2.2 Delivery Status Analysis


In [ ]:
# Delivery status breakdown
if 'Delivery Status' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart
    status_counts = df['Delivery Status'].value_counts()
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
    axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
                colors=colors, explode=[0.05, 0, 0, 0])
    axes[0].set_title('Delivery Status Distribution', fontsize=14, fontweight='bold')

    # By shipping mode
    if 'Shipping Mode' in df.columns:
        crosstab = pd.crosstab(df['Shipping Mode'], df['Delivery Status'], normalize='index') * 100
        crosstab.plot(kind='bar', ax=axes[1], width=0.8)
        axes[1].set_xlabel('Shipping Mode')
        axes[1].set_ylabel('Percentage')
        axes[1].set_title('Delivery Status by Shipping Mode', fontsize=14, fontweight='bold')
        axes[1].legend(title='Status', bbox_to_anchor=(1.02, 1))
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()

print("\n💡 INTERPRETATION:")
print("   • ~55% of deliveries are late - this is our classification target")
print("   • Shipping mode significantly affects delivery performance")


## 2.3 Numeric Features Distribution


In [ ]:
# Key numeric distributions
key_numeric = ['Sales', 'Order Item Quantity', 'Order Item Total', 'Days for shipping (real)']
existing = [c for c in key_numeric if c in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(existing[:4]):
    axes[i].hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(df[col].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df[col].median():.1f}')
    axes[i].set_xlabel(col, fontsize=11)
    axes[i].set_ylabel('Frequency', fontsize=11)
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].legend()

plt.tight_layout()
plt.show()

print("\n💡 INTERPRETATION:")
print("   • Sales and Order Item Total show similar distributions (correlated)")
print("   • Shipping days cluster around 2-6 days")


## 2.4 Correlation Analysis


In [ ]:
# Correlation matrix for key features
numeric_cols = df.select_dtypes(include=[np.number]).columns[:15]  # Top 15 numeric
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with target
if 'Late_delivery_risk' in df.columns:
    target_corr = df[numeric_cols].corrwith(df['Late_delivery_risk']).sort_values(key=abs, ascending=False)
    print("\nTop correlations with Late_delivery_risk:")
    print(target_corr.head(10))

print("\n💡 INTERPRETATION:")
print("   • Days for shipment (scheduled) has strong negative correlation with late delivery")
print("   • Sales features are highly correlated with each other (feature redundancy)")


## 2.5 Customer Segment Analysis


In [ ]:
# Customer segments
if 'Customer Segment' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Segment distribution
    seg_counts = df['Customer Segment'].value_counts()
    axes[0].bar(seg_counts.index, seg_counts.values, color=['#3498db', '#e74c3c', '#2ecc71'])
    axes[0].set_xlabel('Customer Segment')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Orders by Customer Segment', fontweight='bold')

    # Late delivery rate by segment
    if 'Late_delivery_risk' in df.columns:
        late_rate = df.groupby('Customer Segment')['Late_delivery_risk'].mean() * 100
        axes[1].bar(late_rate.index, late_rate.values, color=['#3498db', '#e74c3c', '#2ecc71'])
        axes[1].set_xlabel('Customer Segment')
        axes[1].set_ylabel('Late Delivery Rate (%)')
        axes[1].set_title('Late Delivery Rate by Segment', fontweight='bold')
        axes[1].axhline(df['Late_delivery_risk'].mean()*100, color='red', linestyle='--', label='Overall Average')
        axes[1].legend()

    plt.tight_layout()
    plt.show()

print("\n💡 INTERPRETATION: Consumer segment has highest order volume; late delivery rate similar across segments")


## 2.6 EDA Summary

**Key Insights for Feature Engineering:**
1. Shipping mode is a strong predictor of late delivery
2. Days for shipment (scheduled) has high predictive power
3. Many sales-related features are redundant (high correlation)
4. Customer segment shows similar late delivery rates

**Data Quality Actions Needed:**
- Handle missing values in Order Zipcode, Product Description
- Parse date columns
- Standardize column names
- Handle outliers in numeric features

**Next Step:** → `03_data_preprocessing.ipynb`


In [ ]:
print("✅ EDA complete!")
print("\n➡️ Next: Run 03_data_preprocessing.ipynb")
